# CHIU KNOW? — amostras privadas da voz do Chiu

Este notebook usa o Chatterbox Multilingual V3 localmente na sessão gratuita do Google Colab. A gravação de referência é enviada somente à sessão temporária; ela não é gravada no GitHub.

Antes de executar, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU**.

In [ ]:
!pip -q install chatterbox-tts


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Envie somente Chiu-animada-recorte-final.m4a")
source_name = next(iter(uploaded))
if source_name != "Chiu-animada-recorte-final.m4a":
    raise ValueError("O arquivo precisa ser Chiu-animada-recorte-final.m4a")

reference_wav = Path("chiu_reference_private.wav")
!ffmpeg -loglevel error -y -i "{source_name}" -ac 1 -ar 24000 "{reference_wav}"
print("Referência privada preparada somente nesta sessão.")


In [ ]:
import torch
import torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

if not torch.cuda.is_available():
    raise RuntimeError("Ative uma GPU T4 gratuita antes de continuar.")

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v3")
phrase = "Olá! Eu sou o Chiu. Vamos aprender juntos."
candidates = [
    ("chiu_pt_br_natural.wav", 0.50, 0.45),
    ("chiu_pt_br_animada.wav", 0.65, 0.35),
    ("chiu_pt_br_expressiva.wav", 0.78, 0.30),
]

for output_name, exaggeration, cfg_weight in candidates:
    wav = model.generate(
        phrase,
        language_id="pt",
        audio_prompt_path=str(reference_wav),
        exaggeration=exaggeration,
        cfg_weight=cfg_weight,
    )
    ta.save(output_name, wav.cpu(), model.sr)
    print(output_name)


In [ ]:
from IPython.display import Audio, display

for output_name, _, _ in candidates:
    print(output_name)
    display(Audio(output_name))


In [ ]:
import zipfile

archive = "chiu_voice_candidates_pt_br.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for output_name, _, _ in candidates:
        bundle.write(output_name)
files.download(archive)
